# 1. A cell with two stable states

An ion-level cell expressing a K+ inward rectifier and a depolarizing cation channel
has two stable resting potentials with a saddle between them. We find all three
directly, without simulating a long time course.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tlcdv/bioelectrics/blob/main/notebooks/01-single-cell-attractors.ipynb)

In [ ]:
# On Colab this installs the library. Locally it does nothing if it is already there.
try:
    import bioelectrics  # noqa: F401
except ImportError:
    %pip install -q git+https://github.com/tlcdv/bioelectrics.git

In [ ]:
import torch
from bioelectrics.ionic import IonicTissue, ChannelExpression, K_IR_CAM, CAT_DEP_CAM
from bioelectrics.attractors import find_steady_state

cell = IonicTissue(edges=torch.zeros(2, 0, dtype=torch.long),
                   channels=[ChannelExpression(K_IR_CAM, 1e-16),
                             ChannelExpression(CAT_DEP_CAM, 1e-16)])

### Three start voltages, three steady states

In [ ]:
starts = cell.with_vmem(cell.initial_state((1,), batch=3),
                        torch.tensor([[-0.078], [-0.003], [-0.037]], dtype=torch.float64))
ss = find_steady_state(cell, starts)

for v, stable, slow in zip(ss.vmem[:, 0], ss.stable, ss.slowest_time_s):
    kind = 'stable' if stable else 'saddle'
    hours = '-' if slow is None else '{:.1f} h'.format(slow / 3600)
    print('{:7.1f} mV  {:7s}  slowest time scale {}'.format(1e3 * v, kind, hours))

The middle start lands on the saddle, which is unstable: a small push sends it to one
of the two stable states.

Note the slow time scales. The voltage settles in seconds, but the concentrations keep
moving for hours, because the model has no cell volume regulation. See the
[limitations](https://tlcdv.github.io/bioelectrics/limitations/).

### The current-voltage curve

In [ ]:
import matplotlib.pyplot as plt
from bioelectrics.viz import plot_ionic_iv

fig, ax = plt.subplots(figsize=(6, 4))
plot_ionic_iv(ax, cell)
ax.set_title('Three zero crossings: stable, saddle, stable')
plt.show()